# StockSense AI — Data Profiling

## `01_data_profiling.ipynb`

**Objective:** Establish a reproducible structural profile of the raw StockSense AI datasets before cleaning, transformation, PostgreSQL loading, or modelling.

This notebook intentionally **does not modify the raw data**. It profiles the sales and inventory files and records observations that will feed the formal Data Quality Assessment.

### Questions this notebook answers
- What files and tables are available?
- How large are the datasets?
- What are the columns and data types?
- What is the date coverage?
- How many stores, products and suppliers are represented?
- What are the main categorical dimensions?
- What do sales and inventory distributions look like?
- How much Store × Product coverage overlaps between sales and inventory?

**Next notebook:** `02_data_quality_assessment.ipynb` will perform detailed validation and anomaly investigation.

## 1. Environment & Imports

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

print('Libraries loaded successfully.')

## 2. Define Raw Data Paths

In [ ]:
# Portfolio/GitHub path convention.
# Put the raw CSV files in: data/raw/
PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

SALES_FILE = RAW_DIR / 'retail_sales_ml_apl.csv'
INVENTORY_FILE = RAW_DIR / 'retail_inventory_ml_apl.csv'

print('Project root :', PROJECT_ROOT)
print('Sales file   :', SALES_FILE)
print('Inventory file:', INVENTORY_FILE)

if not SALES_FILE.exists() or not INVENTORY_FILE.exists():
    print('\nWARNING: Place the two CSV files in data/raw/ before running the load cell.')
else:
    print('\nRaw data files found.')

## 3. Load Raw Data

In [ ]:
sales = pd.read_csv(SALES_FILE)
inventory = pd.read_csv(INVENTORY_FILE)

print(f'Sales records    : {len(sales):,}')
print(f'Sales columns    : {sales.shape[1]}')
print(f'Inventory records: {len(inventory):,}')
print(f'Inventory columns: {inventory.shape[1]}')

## 4. Dataset Overview

In [ ]:
overview = pd.DataFrame({
    'Dataset': ['Sales', 'Inventory'],
    'Rows': [len(sales), len(inventory)],
    'Columns': [sales.shape[1], inventory.shape[1]],
    'Memory_MB': [sales.memory_usage(deep=True).sum() / 1024**2,
                  inventory.memory_usage(deep=True).sum() / 1024**2]
})
overview.style.format({'Rows': '{:,.0f}', 'Columns': '{:,.0f}', 'Memory_MB': '{:.2f}'})

## 5. Sales Dataset Structure

In [ ]:
sales.head()

In [ ]:
sales.columns.tolist()

## 6. Inventory Dataset Structure

In [ ]:
inventory.head()

In [ ]:
inventory.columns.tolist()

## 7. Data Types & Basic Statistics

In [ ]:
print('SALES DATA TYPES')
display(sales.dtypes.to_frame('dtype'))

print('INVENTORY DATA TYPES')
display(inventory.dtypes.to_frame('dtype'))

In [ ]:
print('SALES NUMERIC SUMMARY')
display(sales.describe(include='number').T)

print('INVENTORY NUMERIC SUMMARY')
display(inventory.describe(include='number').T)

## 8. Missing-Value Overview

In [ ]:
def missing_summary(df):
    result = pd.DataFrame({
        'Missing_Count': df.isna().sum(),
        'Missing_%': df.isna().mean().mul(100)
    })
    return result.sort_values('Missing_Count', ascending=False)

print('SALES')
display(missing_summary(sales))

print('INVENTORY')
display(missing_summary(inventory))

## 9. Duplicate Overview

In [ ]:
duplicate_summary = pd.DataFrame({
    'Dataset': ['Sales', 'Inventory'],
    'Exact_Duplicates': [sales.duplicated().sum(), inventory.duplicated().sum()],
    'Duplicate_%': [sales.duplicated().mean()*100, inventory.duplicated().mean()*100]
})
duplicate_summary.style.format({'Exact_Duplicates': '{:,.0f}', 'Duplicate_%': '{:.2f}%'})

## 10. Date Coverage

Dates are profiled here without changing the raw DataFrames. A separate cleaning pipeline will later standardise date types.

In [ ]:
sales_dates = pd.to_datetime(sales['Date'], errors='coerce')
inventory_start = pd.to_datetime(inventory['Start Date'], errors='coerce')
inventory_end = pd.to_datetime(inventory['End Date'], errors='coerce')

date_summary = pd.DataFrame({
    'Dataset': ['Sales', 'Inventory Start', 'Inventory End'],
    'Min_Date': [sales_dates.min(), inventory_start.min(), inventory_end.min()],
    'Max_Date': [sales_dates.max(), inventory_start.max(), inventory_end.max()],
    'Distinct_Dates': [sales_dates.nunique(), inventory_start.nunique(), inventory_end.nunique()]
})
date_summary

### Sales Date Distribution

In [ ]:
daily_sales_count = sales.assign(_date=sales_dates).groupby('_date').size()

plt.figure(figsize=(12, 4))
plt.plot(daily_sales_count.index, daily_sales_count.values)
plt.title('Sales Records by Date')
plt.xlabel('Date')
plt.ylabel('Number of Records')
plt.tight_layout()
plt.show()

## 11. Store, Product & Supplier Coverage

In [ ]:
coverage = pd.DataFrame({
    'Dataset': ['Sales', 'Inventory'],
    'Stores': [sales['Store'].nunique(), inventory['Store'].nunique()],
    'Products': [sales['Product'].nunique(), inventory['Product'].nunique()],
    'Suppliers': [sales['Supplier'].nunique(), inventory['Supplier'].nunique()]
})
coverage

### Store Distribution

In [ ]:
store_sales = sales.groupby('Store').size().sort_values(ascending=False)
store_inventory = inventory.groupby('Store').size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
store_sales.plot(kind='bar', ax=ax)
ax.set_title('Sales Records by Store')
ax.set_xlabel('Store')
ax.set_ylabel('Records')
plt.tight_layout()
plt.show()

## 12. Product & Category Structure

In [ ]:
product_cols = [c for c in ['Product', 'Product Description', 'Division', 'Category', 'Subcategory', 'Segment', 'Supplier'] if c in sales.columns]
display(sales[product_cols].drop_duplicates().head(20))

if 'Category' in sales.columns:
    category_counts = sales['Category'].value_counts().head(15)
    display(category_counts.to_frame('Sales_Records'))

## 13. Sales Distribution

In [ ]:
sales_numeric_candidates = [c for c in ['Qty Sold', 'Sales Amount', 'Cogs', 'Returns'] if c in sales.columns]
display(sales[sales_numeric_candidates].describe().T)

In [ ]:
if 'Qty Sold' in sales.columns:
    plt.figure(figsize=(10, 4))
    sales['Qty Sold'].clip(lower=sales['Qty Sold'].quantile(0.01), upper=sales['Qty Sold'].quantile(0.99)).plot(kind='hist', bins=40)
    plt.title('Qty Sold Distribution (1st–99th Percentile Range)')
    plt.xlabel('Qty Sold')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

## 14. Inventory Distribution

In [ ]:
inventory_numeric_candidates = [c for c in ['Qty on hand', 'Stocks Selling Amount', 'Cost of Stocks', 'Stock Unit Selling Price', 'Stock Unit Cost Price'] if c in inventory.columns]
display(inventory[inventory_numeric_candidates].describe().T)

In [ ]:
if 'Qty on hand' in inventory.columns:
    plt.figure(figsize=(10, 4))
    inventory['Qty on hand'].clip(lower=inventory['Qty on hand'].quantile(0.01), upper=inventory['Qty on hand'].quantile(0.99)).plot(kind='hist', bins=40)
    plt.title('Qty on Hand Distribution (1st–99th Percentile Range)')
    plt.xlabel('Qty on Hand')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

## 15. Sales ↔ Inventory Store × Product Coverage

In [ ]:
sales_keys = sales[['Store', 'Product']].drop_duplicates()
inventory_keys = inventory[['Store', 'Product']].drop_duplicates()

coverage_merge = sales_keys.merge(
    inventory_keys,
    on=['Store', 'Product'],
    how='outer',
    indicator=True
)

coverage_result = coverage_merge['_merge'].value_counts().rename_axis('Coverage').reset_index(name='Store_Product_Combinations')
coverage_result['Share_%'] = coverage_result['Store_Product_Combinations'].div(len(coverage_merge)).mul(100)
coverage_result

## 16. Sales Status / Return Profile

In [ ]:
for col in ['Sales Type', 'Promotion/Price Status', 'Reason of Return']:
    if col in sales.columns:
        print(f'--- {col} ---')
        display(sales[col].value_counts(dropna=False).to_frame('Count'))

## 17. Inventory Status Profile

In [ ]:
for col in ['Stock Status']:
    if col in inventory.columns:
        display(inventory[col].value_counts(dropna=False).to_frame('Count'))

## 18. Initial Profiling Findings

After running the notebook, record the key structural observations here. Do **not** make cleaning decisions in this notebook; those belong in `02_data_quality_assessment.ipynb`.

Suggested items to review:
- Dataset scale and grain
- Date coverage and continuity
- Store/Product/Supplier coverage
- Missing columns/fields
- Return and sales-status structure
- Inventory interval structure
- Sales–inventory Store × Product overlap
- Potential anomalies requiring formal quality checks

## 19. Next Step

The next notebook is:

### `02_data_quality_assessment.ipynb`

That notebook will convert the structural profile into a formal quality audit covering missingness, duplicates, negative values, date anomalies, inventory intervals, sales/inventory alignment, financial consistency and forecasting readiness.